# 🎙️ F5-TTS Audiobook (your cloned voice) — GPU

Render a book in your **cloned voice** with F5-TTS on a **free Colab GPU** (fast; on CPU it's ~16× slower).

### Steps
1. **Enable GPU:** `Runtime → Change runtime type → T4 GPU → Save`.
2. Run the cells top-to-bottom.
3. When prompted, **upload** `ref_enhanced.wav` (your enhanced voice reference) and `book.txt` (UTF-8).

Output is written to **Google Drive**, so it **survives disconnects**: if the session drops, just re-run the install + `narrate()` cells and it **resumes** from the last finished chunk.


In [ ]:
# 1) Confirm a GPU is attached (should print a Tesla T4 or similar)
!nvidia-smi -L || echo "NO GPU -> Runtime > Change runtime type > T4 GPU"


In [ ]:
# 2) Install F5-TTS (~2-3 min)
!pip install -q f5-tts soundfile
print("installed")


In [ ]:
# 3) Mount Google Drive (persistent output + resume across disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/f5_audiobook/output', exist_ok=True)
print("Drive mounted -> /content/drive/MyDrive/f5_audiobook/output")


In [ ]:
# 4) Upload your reference audio (ref_enhanced.wav) AND book.txt
from google.colab import files
print("Select ref_enhanced.wav and book.txt ...")
up = files.upload()   # files land in /content
print("uploaded:", list(up.keys()))


In [ ]:
# 5) Narration engine (config + chunking + F5-TTS). Edit CONFIG as needed.
import os, re, glob, time, subprocess, torch
os.environ.setdefault("WANDB_MODE", "disabled")

# ---------------- CONFIG ----------------
BOOK_FILE       = "/content/book.txt"
REFERENCE_AUDIO = "/content/ref_enhanced.wav"
REF_TEXT        = ""                       # transcript of the reference; "" = auto-transcribe (Whisper)
OUTPUT_DIR      = "/content/drive/MyDrive/f5_audiobook/output"
MAX_CHARS       = 200                      # chars per chunk
SENTENCE_PAUSE  = 0.30
PARAGRAPH_PAUSE = 0.65
NFE_STEP        = 32                        # higher = better/slower (16 ~2x faster)
SPEED           = 1.0
MAKE_MP3        = True
# ----------------------------------------
CHUNK_DIR = os.path.join(OUTPUT_DIR, "chunks")

def split_sentences(p):
    p = re.sub(r"\s+", " ", p).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?\u2026\u3002\uff01\uff1f])\s*", p) if s.strip()] if p else []

def split_long(s, m):
    if len(s) <= m: return [s]
    out, buf = [], ""
    for part in re.split(r"(?<=[,;:\uff0c\uff1b\uff1a\u3001])\s*", s):
        if len(part) > m:
            if " " in part:
                for w in part.split(" "):
                    if len(buf)+len(w)+1 > m and buf: out.append(buf.strip()); buf=""
                    buf += w+" "
            else:
                if buf.strip(): out.append(buf.strip()); buf=""
                for i in range(0,len(part),m): out.append(part[i:i+m])
            continue
        if len(buf)+len(part)+1 > m and buf: out.append(buf.strip()); buf=""
        buf += part+" "
    if buf.strip(): out.append(buf.strip())
    return out

def build_chunks(text):
    chunks = []
    for para in [x for x in re.split(r"\n\s*\n", text) if x.strip()]:
        buf, pc = "", []
        for sent in split_sentences(para):
            for piece in split_long(sent, MAX_CHARS):
                if len(buf)+len(piece)+1 > MAX_CHARS and buf: pc.append(buf.strip()); buf=""
                buf += piece+" "
        if buf.strip(): pc.append(buf.strip())
        for i,c in enumerate(pc): chunks.append((c, PARAGRAPH_PAUSE if i==len(pc)-1 else SENTENCE_PAUSE))
    return chunks

_M = {}
def _load():
    if _M: return _M["f5"], _M["ra"], _M["rt"]
    from f5_tts.api import F5TTS
    from f5_tts.infer.utils_infer import preprocess_ref_audio_text
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", dev, "|", (torch.cuda.get_device_name(0) if dev=="cuda" else "CPU (slow!)"))
    f5 = F5TTS(device=dev)
    ra, rt = preprocess_ref_audio_text(REFERENCE_AUDIO, REF_TEXT)
    print("ref_text:", repr(rt[:100]))
    _M.update(f5=f5, ra=ra, rt=rt)
    return f5, ra, rt

def narrate(test_limit=0):
    import numpy as np, soundfile as sf
    assert os.path.isfile(BOOK_FILE), BOOK_FILE + " not found (upload it)"
    assert os.path.isfile(REFERENCE_AUDIO), REFERENCE_AUDIO + " not found (upload it)"
    os.makedirs(CHUNK_DIR, exist_ok=True)
    chunks = build_chunks(open(BOOK_FILE, encoding="utf-8").read())
    todo = chunks if test_limit<=0 else chunks[:test_limit]
    print(str(len(chunks)) + " chunks total; rendering " + str(len(todo)))
    f5, ra, rt = _load()
    t0, done = time.time(), 0
    for i,(ct,pause) in enumerate(todo,1):
        outp = os.path.join(CHUNK_DIR, "chunk_%05d.wav" % i)
        if os.path.isfile(outp) and os.path.getsize(outp)>0: continue
        wav, sr, _ = f5.infer(ref_file=ra, ref_text=rt, gen_text=ct, nfe_step=NFE_STEP, speed=SPEED, show_info=lambda *a, **k: None)
        wav = np.asarray(wav, dtype=np.float32)
        if pause>0: wav = np.concatenate([wav, np.zeros(int(sr*pause), dtype=np.float32)])
        sf.write(outp, wav, sr, subtype="PCM_16")
        done += 1; avg = (time.time()-t0)/done
        print("[%d/%d] %dc  %.1fs/chunk  ETA %.1f min" % (i, len(todo), len(ct), avg, avg*(len(todo)-i)/60))
    paths = sorted(glob.glob(os.path.join(CHUNK_DIR, "chunk_*.wav")))
    if test_limit>0: paths = paths[:test_limit]
    lst = os.path.join(OUTPUT_DIR, "_concat.txt")
    open(lst, "w").write("".join("file '" + os.path.abspath(p) + "'\n" for p in paths))
    subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",lst,"-c","copy",os.path.join(OUTPUT_DIR,"audiobook.wav")], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if MAKE_MP3:
        subprocess.run(["ffmpeg","-y","-f","concat","-safe","0","-i",lst,"-c:a","libmp3lame","-q:a","4",os.path.join(OUTPUT_DIR,"audiobook.mp3")], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("DONE -> " + os.path.join(OUTPUT_DIR, "audiobook.mp3"))

print("ready. Run narrate(test_limit=5) for a smoke test, then narrate(test_limit=0) for the whole book.")


In [ ]:
# 6) Smoke test — render the first 5 chunks and sanity-check the voice/quality
narrate(test_limit=5)


In [ ]:
# 7) Full book (auto-resumes if the session disconnects — just re-run this cell)
narrate(test_limit=0)


In [ ]:
# 8) Download the finished audiobook
from google.colab import files
files.download('/content/drive/MyDrive/f5_audiobook/output/audiobook.mp3')
